In [1]:
######################################################################################################################
# Notebook: 01_Construccion_Corpus_NANDINA.ipynb
# Autor del código: Vladimir Molleapasa Gutierrez
# Fecha de generación: 13/01/2026
# Código generado con asistencia de ChatGPT 5.2 Thinking
# Prompt original: "Adjunto el PDF de la NANDINA (Decisión 885). Genera un notebook 01_Construccion_Corpus_NANDINA 
#                   que extraiga códigos 4/6/8 dígitos (con puntos), descripciones y U.F. (multilínea), 
#                   capture contexto Sección/Capítulo, exporte nandina_corpus.jsonl + run_metadata.json 
#                   (SHA-256/entorno/parámetros) + summary.csv (QA). Documenta orientado a reproducibilidad"
# Autor del prompt: Vladimir Molleapasa Gutierrez
# Ajustes y validación: Vladimir Molleapasa Gutierrez
# Uso académico, con revisión propia del autor.
# Licencia: Uso académico, no comercial.
######################################################################################################################

# =============================================================================
# Objetivo:
#   Transformar el PDF "CAN Decisión 885 - NANDINA (Gaceta 4359)" en un corpus
#   estructurado (JSONL/JSON) para experimentos reproducibles (PLN/IR/clasificación).
#
# Principios de reproducibilidad incorporados:
#   - Trazabilidad del insumo: hash SHA-256 del PDF.
#   - Captura del entorno: versión de Python y librerías relevantes.
#   - Especificación explícita de entradas/salidas y del esquema de datos.
#   - Logs y agregados para validación (conteos por nivel).
#
# Supuestos derivados del documento:
#   - La NANDINA usa códigos de 8 dígitos (con separadores por puntos en el Anexo).
#   - El Anexo presenta filas tipo: "Código  Descripción  U.F."
#   - En algunas páginas, la U.F. puede quedar en una línea separada por salto de línea.
# =============================================================================

from __future__ import annotations

import json
import re
import hashlib
import platform
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from typing import Iterable, Iterator, Optional, List, Dict, Any, Tuple

# Dependencia principal para extracción de texto con layout razonable en PDFs "text-based".
# Instalación (si aplica): pip install pdfplumber
import pdfplumber

# Captura de versiones de librerías para trazabilidad (Python >= 3.8).
try:
    from importlib.metadata import version as pkg_version
except Exception:  # pragma: no cover
    pkg_version = None


In [2]:
# -----------------------------------------------------------------------------
# Configuración del experimento (editar según entorno local)
# -----------------------------------------------------------------------------

# Raíz del proyecto 
PROJECT_ROOT = Path(r"C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código")

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

PDF_PATH = DATA_RAW / "CAN Desición 885 - Nanadina Gaceta 4359.pdf"

# Salidas (procesados) dentro del repo
OUTPUT_DIR = DATA_PROCESSED
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_JSONL = OUTPUT_DIR / "nandina_corpus.jsonl"
OUT_HIERARCHY_JSON = OUTPUT_DIR / "nandina_hierarchy.json"
OUT_METADATA = OUTPUT_DIR / "run_metadata.json"
OUT_SUMMARY = OUTPUT_DIR / "summary.csv"

assert PDF_PATH.exists(), f"No se encuentra el PDF en: {PDF_PATH}"

print("PDF:", PDF_PATH)
print("JSONL:", OUT_JSONL)
print("Hierarchy:", OUT_HIERARCHY_JSON)
print("Metadata:", OUT_METADATA)
print("Summary:", OUT_SUMMARY)


PDF: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\raw\CAN Desición 885 - Nanadina Gaceta 4359.pdf
JSONL: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\nandina_corpus.jsonl
Hierarchy: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\nandina_hierarchy.json
Metadata: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\run_metadata.json
Summary: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\summary.csv


In [3]:
# -----------------------------------------------------------------------------
# Esquema de datos (salida)
# -----------------------------------------------------------------------------
@dataclass
class NandinaRecord:
    """
    Registro atómico extraído del Anexo NANDINA.

    Campos mínimos para un "corpus" utilizable:
      - code_raw: Código tal como aparece en el PDF (p. ej., "3201.10.00").
      - code_digits: Solo dígitos (p. ej., "32011000").
      - level: Nivel inferido por longitud (4=partida, 6=subpartida SA, 8=subpartida NANDINA).
      - description: Texto normalizado (sin U.F.).
      - unit: Unidad física (si se identifica).
      - context: Sección/Capítulo cuando se detectan en el flujo del documento.
      - source: Página y línea para auditoría.

    Nota:
      - Este registro NO intenta reconstruir jerarquía completa; conserva contexto suficiente
        para rearmarla posteriormente (p. ej., con un post-proceso por niveles).
    """
    code_raw: str
    code_digits: str
    level: str
    description: str
    unit: Optional[str]

    section: Optional[str]
    section_title: Optional[str]
    chapter: Optional[str]
    chapter_title: Optional[str]

    page: int
    line_no_on_page: int
    line_text: str

In [4]:
# -----------------------------------------------------------------------------
# Utilidades de trazabilidad y entorno
# -----------------------------------------------------------------------------
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    """Calcula SHA-256 del archivo para trazabilidad del insumo."""
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def environment_metadata() -> Dict[str, Any]:
    """Captura metadatos del entorno para reproducibilidad."""
    libs = {}
    if pkg_version is not None:
        for lib in ["pdfplumber", "pdfminer.six"]:
            try:
                libs[lib] = pkg_version(lib)
            except Exception:
                libs[lib] = None

    return {
        "timestamp_utc": datetime.utcnow().isoformat() + "Z",
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "libraries": libs,
    }

In [5]:
# -----------------------------------------------------------------------------
# Extracción de líneas del PDF (con posición: página y orden)
# -----------------------------------------------------------------------------
def iter_pdf_lines(pdf_path: Path, max_pages: Optional[int] = None) -> Iterator[Tuple[int, int, str]]:
    """
    Itera líneas de texto extraídas del PDF.

    Retorna tuplas:
      (page_number_1based, line_index_1based, line_text)

    Consideraciones:
      - pdfplumber extrae texto por página; el orden de líneas suele ser estable
        en PDFs "text-based" como el Anexo de la NANDINA.
      - Si una página no retorna texto, se omite. Esto evita fallas en PDFs con páginas-imagen.
    """
    with pdfplumber.open(str(pdf_path)) as pdf:
        n_pages = len(pdf.pages)
        last = n_pages if max_pages is None else min(max_pages, n_pages)

        for i in range(last):
            page_no = i + 1
            text = pdf.pages[i].extract_text()
            if not text:
                continue

            # Normalización ligera: recortar y conservar líneas no vacías.
            raw_lines = [ln.strip() for ln in text.splitlines()]
            lines = [ln for ln in raw_lines if ln]

            for j, ln in enumerate(lines, start=1):
                yield page_no, j, ln


In [6]:
# -----------------------------------------------------------------------------
# Parser de NANDINA: detección de contexto (sección/capítulo) y extracción de códigos
# -----------------------------------------------------------------------------
SECTION_RE = re.compile(r"^SECCI[ÓO]N\s+([IVXLC]+)\b", re.IGNORECASE)
CHAPTER_RE = re.compile(r"^Cap[ií]tulo\s+(\d{1,2})\b", re.IGNORECASE)

# Códigos típicos observados en el Anexo:
#  - 8 dígitos con puntos: 3201.10.00
#  - 6 dígitos con punto: 3201.90
#  - 4 dígitos con punto: 32.01
CODE_RE = re.compile(r"^(?P<code>\d{2}\.\d{2}|\d{4}\.\d{2}(?:\.\d{2})?)\b")

# Unidades probables (U.F.). Lista acotada con patrones comunes.
# Puede ampliarse si se detectan unidades adicionales durante validación.
UNIT_TOKEN_RE = re.compile(r"^[A-Za-zµ°%]{1,4}\d{0,2}$")


def infer_level(code_digits: str) -> str:
    """Inferencia del nivel por longitud de dígitos."""
    if len(code_digits) == 8:
        return "nandina_8d"
    if len(code_digits) == 6:
        return "hs_6d"
    if len(code_digits) == 4:
        return "partida_4d"
    return f"unknown_{len(code_digits)}d"


def split_description_and_unit(text: str) -> Tuple[str, Optional[str]]:
    """
    Separa descripción y unidad si la unidad aparece como último token.

    Regla:
      - Si el último token parece una U.F. (patrón corto) => se extrae.
      - En caso contrario => unit=None.
    """
    tokens = text.strip().split()
    if not tokens:
        return "", None

    last = tokens[-1]
    if UNIT_TOKEN_RE.match(last) and len(last) <= 5:
        desc = " ".join(tokens[:-1]).strip()
        unit = last
        return desc, unit

    return text.strip(), None


def normalize_desc(rest: str) -> str:
    """
    Normaliza el texto descriptivo conservando semántica.
    - Elimina prefijos de guiones usados como indentación ("- ", "- - ").
    - Colapsa espacios múltiples.
    """
    rest = rest.strip()
    rest = re.sub(r"^-\s*-\s*", "", rest)  # "- - ..."
    rest = re.sub(r"^-\s*", "", rest)      # "- ..."
    rest = re.sub(r"\s+", " ", rest)
    return rest.strip()


def parse_nandina_records(lines: Iterable[Tuple[int, int, str]]) -> List[NandinaRecord]:
    """
    Extrae registros NANDINA a partir de líneas del PDF, preservando contexto.

    Manejo de multilínea:
      - Si se detecta un código sin unidad y la descripción parece "abierta",
        se acumulan líneas siguientes hasta:
          (a) detectar una unidad en línea aislada, o
          (b) detectar un nuevo código (se cierra el registro anterior).
    """
    records: List[NandinaRecord] = []

    section: Optional[str] = None
    section_title: Optional[str] = None
    chapter: Optional[str] = None
    chapter_title: Optional[str] = None

    pending: Optional[NandinaRecord] = None

    # Para capturar títulos en la línea posterior a SECCIÓN / Capítulo:
    expect_section_title = False
    expect_chapter_title = False

    for page, line_no, line in lines:
        # 1) Contexto: SECCIÓN
        msec = SECTION_RE.match(line)
        if msec:
            section = msec.group(1).upper()
            section_title = None
            expect_section_title = True
            # Si existía un registro pendiente, se cierra antes de cambiar contexto.
            if pending is not None:
                records.append(pending)
                pending = None
            continue

        if expect_section_title:
            # La siguiente línea útil se interpreta como título de sección.
            section_title = line.strip()
            expect_section_title = False
            continue

        # 2) Contexto: Capítulo
        mcap = CHAPTER_RE.match(line)
        if mcap:
            chapter = mcap.group(1).zfill(2)
            chapter_title = None
            expect_chapter_title = True
            if pending is not None:
                records.append(pending)
                pending = None
            continue

        if expect_chapter_title:
            chapter_title = line.strip()
            expect_chapter_title = False
            continue

        # 3) Detección de código (4/6/8 dígitos con puntos)
        mcode = CODE_RE.match(line)
        if mcode:
            # Si había un registro pendiente, se cierra al detectar un nuevo código.
            if pending is not None:
                records.append(pending)
                pending = None

            code_raw = mcode.group("code")
            code_digits = re.sub(r"\D", "", code_raw)
            level = infer_level(code_digits)

            rest = line[len(code_raw):].strip()
            rest = normalize_desc(rest)

            # Separación descripción / unidad en la misma línea
            desc, unit = split_description_and_unit(rest)

            rec = NandinaRecord(
                code_raw=code_raw,
                code_digits=code_digits,
                level=level,
                description=desc,
                unit=unit,
                section=section,
                section_title=section_title,
                chapter=chapter,
                chapter_title=chapter_title,
                page=page,
                line_no_on_page=line_no,
                line_text=line,
            )

            # Heurística de multilínea:
            # - Si no hay unidad y el texto parece continuar (por longitud o termina en coma/punto y coma)
            # - Se deja como "pending" para concatenar líneas siguientes.
            if unit is None and (desc.endswith((",", ";", ":", "-", "(", "de")) or len(desc) > 40):
                pending = rec
            else:
                records.append(rec)

            continue

        # 4) Continuación multilínea (si existe registro pendiente)
        if pending is not None:
            # Caso A: la línea es SOLO una unidad (p. ej., "kg") => cerrar registro
            candidate = line.strip()
            if UNIT_TOKEN_RE.match(candidate) and len(candidate) <= 5:
                pending.unit = candidate
                records.append(pending)
                pending = None
                continue

            # Caso B: concatenar a la descripción (hasta nuevo código o unidad)
            pending.description = normalize_desc(pending.description + " " + candidate)
            pending.line_text = pending.line_text + " | " + line  # auditoría del "join"
            continue

        # 5) Líneas sin relevancia directa para el corpus (se ignoran)
        #    El diseño deliberadamente evita forzar parseo de notas y reglas generales
        #    para mantener un corpus "código -> descripción" más limpio. Si se requiere,
        #    esas secciones pueden incorporarse en una fase posterior.
        continue

    # Cierre final del registro pendiente
    if pending is not None:
        records.append(pending)

    return records


In [7]:
# -----------------------------------------------------------------------------
# Ejecución del pipeline y escritura de artefactos
# -----------------------------------------------------------------------------
def write_jsonl(records: List[NandinaRecord], path: Path) -> None:
    """Escribe JSONL (una línea por registro) con UTF-8."""
    with path.open("w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(asdict(r), ensure_ascii=False) + "\n")


def write_run_metadata(pdf_path: Path, out_path: Path, params: Dict[str, Any]) -> None:
    """Escribe metadatos de ejecución (insumo + entorno + parámetros)."""
    meta = {
        "input_pdf": {
            "path": str(pdf_path),
            "name": pdf_path.name,
            "sha256": sha256_file(pdf_path),
        },
        "environment": environment_metadata(),
        "parameters": params,
    }
    out_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")


def write_summary_csv(records: List[NandinaRecord], path: Path) -> None:
    """Genera un CSV mínimo para validación (conteos por nivel)."""
    from collections import Counter

    c = Counter(r.level for r in records)
    lines = ["level,count"]
    for k in sorted(c.keys()):
        lines.append(f"{k},{c[k]}")
    path.write_text("\n".join(lines), encoding="utf-8")

In [8]:
# =============================================================================
# MAIN — Construcción del corpus NANDINA desde PDF (Decisión 885)
#
# Entradas:
#   - PDF_PATH: PDF en /data/raw
#
# Salidas (en /data/processed):
#   - nandina_corpus.jsonl      : registros atómicos (código/descr./U.F./contexto/página)
#   - run_metadata.json         : trazabilidad (hash PDF, entorno, parámetros)
#   - summary.csv               : control de calidad (conteos por nivel)
#
# Reproducibilidad:
#   - El hash SHA-256 del PDF queda registrado en run_metadata.json.
#   - Versiones de librerías y parámetros del run también quedan registradas.
# =============================================================================

# Parámetro de control de ejecución:
# - None: procesa todas las páginas del PDF
# - int : procesa solo las primeras N páginas (útil para pruebas)
MAX_PAGES = None

# 1) Validación de insumos y directorios
assert PDF_PATH.exists(), f"No se encuentra el PDF en: {PDF_PATH}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 2) Parámetros del run (quedan persistidos en run_metadata.json)
params = {
    "max_pages": MAX_PAGES,                       # None = todas las páginas
    "input_pdf": PDF_PATH.name,
    "output_dir": str(OUTPUT_DIR),
    "outputs": {
        "jsonl": str(OUT_JSONL),
        "hierarchy_json": str(OUT_HIERARCHY_JSON),
        "metadata": str(OUT_METADATA),
        "summary": str(OUT_SUMMARY),
    },
    "parser": {
        "engine": "pdfplumber",
        "strategy": "regex-based row detection + multiline join for description/unit",
        "code_patterns": ["NN.NN", "NNNN.NN", "NNNN.NN.NN"],   # 4/6/8 dígitos con puntos
        "unit_detection": "last-token or standalone-line (U.F.)",
    },
}

# 3) Extracción de texto del PDF (página → líneas)
pdf_lines = list(iter_pdf_lines(PDF_PATH, max_pages=MAX_PAGES))

# 4) Parsing: líneas → registros estructurados
records = parse_nandina_records(pdf_lines)

# 5) Escritura de artefactos reproducibles
write_jsonl(records, OUT_JSONL)
write_run_metadata(PDF_PATH, OUT_METADATA, params)
write_summary_csv(records, OUT_SUMMARY)

# 6) Jerarquía (opcional): se escribe solo si la función existe y retorna estructura válida
if "build_hierarchy" in globals():
    hierarchy = build_hierarchy(records)  # debe retornar dict serializable
    with open(OUT_HIERARCHY_JSON, "w", encoding="utf-8") as f:
        json.dump(hierarchy, f, ensure_ascii=False, indent=2)

# 7) Reporte mínimo en consola (sanity check)
print("=== Corpus NANDINA generado ===")
print(f"Registros extraídos: {len(records)}")
print(f"JSONL (corpus):      {OUT_JSONL.resolve()}")
print(f"Metadata (run):      {OUT_METADATA.resolve()}")
print(f"Summary (QA):        {OUT_SUMMARY.resolve()}")
if (OUTPUT_DIR / OUT_HIERARCHY_JSON.name).exists():
    print(f"Jerarquía (JSON):    {OUT_HIERARCHY_JSON.resolve()}")


=== Corpus NANDINA generado ===
Registros extraídos: 9785
JSONL (corpus):      C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\nandina_corpus.jsonl
Metadata (run):      C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\run_metadata.json
Summary (QA):        C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\summary.csv
